# 1. Dataset Structure and Ring Identification

Before preprocessing, we identify the available participant files,
ring types, and file integrity.

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
DATA_PATH = Path("../data/raw")


pkl_files = sorted(DATA_PATH.glob("*.pkl"))

print("Total pickle files:", len(pkl_files))

print("\nFirst 15 filenames:")
for file in pkl_files[:15]:
    print(file.name, "| Size:", file.stat().st_size, "bytes")

print("\nRing-wise file counts:")

ring1_files = [f for f in pkl_files if "ring1" in f.name.lower()]
ring2_files = [f for f in pkl_files if "ring2" in f.name.lower()]

print("Ring 1:", len(ring1_files))
print("Ring 2:", len(ring2_files))

print("\nEmpty files:")
for file in pkl_files:
    if file.stat().st_size == 0:
        print(file.name)


Total pickle files: 57

First 15 filenames:
00000_ring1_processed.pkl | Size: 37186036 bytes
00000_ring2_processed.pkl | Size: 9955304 bytes
00001_ring1_processed.pkl | Size: 39033475 bytes
00001_ring2_processed.pkl | Size: 19639708 bytes
00002_ring1_processed.pkl | Size: 77412719 bytes
00002_ring2_processed.pkl | Size: 66551040 bytes
00003_ring1_processed.pkl | Size: 42246537 bytes
00003_ring2_processed.pkl | Size: 11980420 bytes
00004_ring1_processed.pkl | Size: 43510740 bytes
00004_ring2_processed.pkl | Size: 28799258 bytes
00005_ring1_processed.pkl | Size: 59909284 bytes
00005_ring2_processed.pkl | Size: 34892751 bytes
00006_ring1_processed.pkl | Size: 59725604 bytes
00006_ring2_processed.pkl | Size: 35897837 bytes
00007_ring1_processed.pkl | Size: 44025664 bytes

Ring-wise file counts:
Ring 1: 28
Ring 2: 29

Empty files:
00029_ring2_processed.pkl


In [6]:

import re
from pathlib import Path
import pandas as pd

# Extract participant IDs from filenames
ring_mapping = {}

for file in pkl_files:
    match = re.match(r"(\d+)_ring([12])_processed\.pkl", file.name)

    if match:
        participant_id = match.group(1)
        ring_number = int(match.group(2))

        if participant_id not in ring_mapping:
            ring_mapping[participant_id] = {}

        ring_mapping[participant_id][f"ring{ring_number}"] = file

# Create participant mapping table
mapping_rows = []

for participant_id, rings in sorted(ring_mapping.items()):
    r1 = rings.get("ring1")
    r2 = rings.get("ring2")

    mapping_rows.append({
        "participant_id": participant_id,
        "ring1_available": r1 is not None,
        "ring2_available": r2 is not None,
        "ring1_size_MB": round(r1.stat().st_size / (1024**2), 2) if r1 else None,
        "ring2_size_MB": round(r2.stat().st_size / (1024**2), 2) if r2 else None,
        "ring2_empty": r2.stat().st_size == 0 if r2 else False
    })

ring_mapping_df = pd.DataFrame(mapping_rows)

display(ring_mapping_df)

print("\nParticipant mapping summary:")
print("Total participant IDs:", len(ring_mapping_df))
print("Both rings available:", 
      (ring_mapping_df["ring1_available"] & ring_mapping_df["ring2_available"]).sum())

print("Only Ring 1:", 
      (ring_mapping_df["ring1_available"] & ~ring_mapping_df["ring2_available"]).sum())

print("Only Ring 2:", 
      (~ring_mapping_df["ring1_available"] & ring_mapping_df["ring2_available"]).sum())


,participant_id,ring1_available,ring2_available,ring1_size_MB,ring2_size_MB,ring2_empty
0,00000,True,True,35.46,9.49,False
1,00001,True,True,37.23,18.73,False
2,00002,True,True,73.83,63.47,False
3,00003,True,True,40.29,11.43,False
4,00004,True,True,41.50,27.47,False
5,00005,True,True,57.13,33.28,False
6,00006,True,True,56.96,34.23,False
7,00007,True,True,41.99,29.01,False
8,00008,True,True,36.15,28.65,False
9,00009,True,True,39.56,34.31,False



Participant mapping summary:
Total participant IDs: 29
Both rings available: 28
Only Ring 1: 0
Only Ring 2: 1


In [7]:
pairing_rows = []

for participant_id, rings in sorted(ring_mapping.items()):

    ring1_file = rings.get("ring1")
    ring2_file = rings.get("ring2")

    # Skip participants without both files
    if ring1_file is None or ring2_file is None:
        continue

    # Skip empty files
    if ring1_file.stat().st_size == 0 or ring2_file.stat().st_size == 0:
        continue

    try:
        ring1_df = pd.read_pickle(ring1_file)
        ring2_df = pd.read_pickle(ring2_file)

        # Check that timestamp columns exist
        if not {"start", "end"}.issubset(ring1_df.columns):
            print(f"Missing timestamps in Ring 1: {participant_id}")
            continue

        if not {"start", "end"}.issubset(ring2_df.columns):
            print(f"Missing timestamps in Ring 2: {participant_id}")
            continue

        # Convert timestamps to numeric values
        r1_start = pd.to_numeric(ring1_df["start"], errors="coerce").dropna()
        r1_end = pd.to_numeric(ring1_df["end"], errors="coerce").dropna()

        r2_start = pd.to_numeric(ring2_df["start"], errors="coerce").dropna()
        r2_end = pd.to_numeric(ring2_df["end"], errors="coerce").dropna()

        if min(len(r1_start), len(r1_end), len(r2_start), len(r2_end)) == 0:
            continue

        # Overall recording ranges
        ring1_min = r1_start.min()
        ring1_max = r1_end.max()

        ring2_min = r2_start.min()
        ring2_max = r2_end.max()

        # Calculate overlap between overall time ranges
        overlap_start = max(ring1_min, ring2_min)
        overlap_end = min(ring1_max, ring2_max)

        overlap_seconds = max(0, overlap_end - overlap_start)

        pairing_rows.append({
            "participant_id": participant_id,
            "ring1_windows": len(ring1_df),
            "ring2_windows": len(ring2_df),
            "ring1_start": ring1_min,
            "ring1_end": ring1_max,
            "ring2_start": ring2_min,
            "ring2_end": ring2_max,
            "overlap_seconds": overlap_seconds,
            "overlap_minutes": round(overlap_seconds / 60, 2),
            "overall_time_overlap": overlap_seconds > 0
        })

    except Exception as e:
        print(f"Error for participant {participant_id}: {e}")

pairing_df = pd.DataFrame(pairing_rows)

display(pairing_df)

print("\nPairing summary:")
print("Participants checked:", len(pairing_df))
print("Participants with overlapping overall time ranges:",
      pairing_df["overall_time_overlap"].sum())
print("Participants without overlapping overall time ranges:",
      (~pairing_df["overall_time_overlap"]).sum())

,participant_id,ring1_windows,ring2_windows,ring1_start,ring1_end,ring2_start,ring2_end,overlap_seconds,overlap_minutes,overall_time_overlap
0,00000,38,16,1.742608e+09,1.742609e+09,1.742608e+09,1.742609e+09,479.983492,8.00,True
1,00001,40,20,1.742458e+09,1.742460e+09,1.742458e+09,1.742458e+09,600.001359,10.00,True
2,00002,79,68,1.742608e+09,1.742994e+09,1.742992e+09,1.742994e+09,2017.061819,33.62,True
3,00003,43,20,1.742611e+09,1.742613e+09,1.742611e+09,1.742613e+09,1709.753218,28.50,True
4,00004,45,30,1.742615e+09,1.742617e+09,1.742615e+09,1.742617e+09,2009.818570,33.50,True
5,00005,61,36,1.742621e+09,1.742624e+09,1.742621e+09,1.742624e+09,2699.980074,45.00,True
6,00006,61,41,1.742628e+09,1.742647e+09,1.742628e+09,1.742647e+09,18926.051244,315.43,True
7,00007,45,33,1.742632e+09,1.742634e+09,1.742632e+09,1.742634e+09,1829.842787,30.50,True
8,00008,39,37,1.742639e+09,1.742641e+09,1.742639e+09,1.742641e+09,2069.960847,34.50,True
9,00009,43,43,1.742643e+09,1.742645e+09,1.742643e+09,1.742645e+09,2009.880825,33.50,True



Pairing summary:
Participants checked: 27
Participants with overlapping overall time ranges: 27
Participants without overlapping overall time ranges: 0
